# Swarm recordings — exploration

Interactive companion to the scripts in `Analysis/`. Nothing here reimplements a measure: hull area
and wall contacts are read from the recordings, exactly as the scripts do.

**Kernel:** select your `PythonAnalysis` conda environment (top right in VS Code). It needs
`ipykernel`, `matplotlib`, `numpy`, `pandas`, and `scipy` for the validation section.

In [ ]:
import sys
from pathlib import Path

# The scripts live one level up, in Analysis/. Make them importable from this notebook.
ANALYSIS = Path.cwd().parent
if str(ANALYSIS) not in sys.path:
    sys.path.insert(0, str(ANALYSIS))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

import plot_trajectory as pt
import plot_grid as pg

print("analysis modules loaded from", ANALYSIS)

## 1. Point at a batch

`RECORDINGS` is the folder of `.json` files a batch wrote. Everything below works off it.

In [ ]:
RECORDINGS = ANALYSIS.parent / "Assets/SimulationRecordings/Combinations/20260817_004421"

runs = pg.load_runs(RECORDINGS)
print(f"{len(runs)} recordings in {RECORDINGS.name}")

rows, cols, hue, levels = pg.choose_axes(runs, None, None, None)
print(f"rows {pg.SHORT[rows]} {levels[rows]}")
print(f"cols {pg.SHORT[cols]} {levels[cols]}")
print(f"line {pg.SHORT[hue]} {levels[hue]}")

## 2. The sweep as a table

One row per run, so you can sort and filter before deciding what to look at closely.

In [ ]:
table = pd.DataFrame([{
    "file": r["path"].stem,
    "type": r["header"].get("swarmType"),
    "random": r["randomMovement"],
    "perception": r["perceptionRadius"],
    "maxSpeed": r["maxSpeed"],
    "agents": r["header"].get("agentCount"),
    "duration": round(float(r["times"][-1]), 2),
    "hull_start": round(float(r["areas"][0]), 1),
    "hull_end": round(float(r["areas"][-1]), 1),
    "hull_ratio": round(float(r["areas"][-1] / r["areas"][0]), 2) if r["areas"][0] > 0 else np.nan,
    "peak_contacts": int(r["contacts"].max()),
    "unique_contacts": int(r["unique"][-1]),
    "ended": r["header"].get("endReason"),
} for r in runs])

table.sort_values(["perception", "random", "maxSpeed"]).reset_index(drop=True)

## 3. The whole sweep on one page

Small multiples: the two coarser factors are the grid, the third is colour inside each panel.
Shared axes, so differences read as distances.

In [ ]:
title = f"{runs[0]['header'].get('swarmType')}  ·  {len(runs)} runs"

fig = pg.grid_figure(runs, rows, cols, hue, levels, "hull", False, title + "\nhull area over time")
plt.show()

In [ ]:
fig = pg.grid_figure(runs, rows, cols, hue, levels, "contacts", False,
                     title + "\nagents touching a wall over time")
plt.show()

In [ ]:
fig = pg.summary_figure(runs, rows, cols, hue, levels, title + "\nsweep summary")
plt.show()

## 4. One run in detail

Pick a row from the table above by filename, or index into `runs`.

In [ ]:
run = runs[0]
data = pt.load_trajectory(run["path"])
times, areas, contacts, cumulative = pt.analyse(data)

fig = pt.build_figure(run["path"], data, times, areas, contacts, cumulative)
plt.show()

## 5. Compare a chosen subset

Overlay any selection of runs directly, when the grid is more than you need.

In [ ]:
subset = [r for r in runs if r["maxSpeed"] == 4.0 and r["randomMovement"] == 0.0]

fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
for r in sorted(subset, key=lambda r: r["perceptionRadius"]):
    ax.plot(r["times"], r["areas"], linewidth=1.8, label=f"perception {r['perceptionRadius']:g}")

ax.set_xlabel("time (s)")
ax.set_ylabel("hull area (u²)")
ax.grid(alpha=0.25)
ax.legend()
ax.set_title(f"{len(subset)} runs at maxSpeed 4, random 0")
plt.show()

## 6. Validate the recorded hull areas

Independent check against `scipy.spatial.ConvexHull`. Needs `scipy` in the environment.

Frames flagged as *trim ties* are ones where the file's three decimal places cannot settle which
agent the trim discards — not errors. See the README for the detail.

In [ ]:
import validate_hull as vh

ok = all(vh.validate(r["path"], tolerance=2e-3, decimals=3) for r in runs[:4])
print("\nall checked files passed" if ok else "\nsome files failed")

## 7. Save figures to disk

Same output as running `plot_grid.py` from a terminal.

In [ ]:
out_dir = RECORDINGS / "figures"
out_dir.mkdir(parents=True, exist_ok=True)

for metric, name in (("hull", "hull_grid"), ("contacts", "contacts_grid")):
    fig = pg.grid_figure(runs, rows, cols, hue, levels, metric, False, title)
    fig.savefig(out_dir / f"{runs[0]['header'].get('swarmType','runs').lower()}_{name}.png", dpi=140)
    plt.close(fig)

print("written to", out_dir)